Projet Big Data : marché de l'emploi des métiers de la data
Notebook 1 : web scraping et collecte des données

Ce notebook correspond à l'étape 3 du sujet (acquisition). Il collecte des offres d'emploi des métiers de la data sur deux sources :
- l'API officielle de France Travail, interrogée avec la bibliothèque requests ;
- le site Welcome to the Jungle (WTTJ), scrapé avec Selenium et BeautifulSoup.

Pour chaque offre, on récupère des informations structurées (intitulé, contrat, salaire, lieu, expérience, formation) et textuelles (description). Les deux sources sont mises au même format de 20 colonnes, puis enregistrées dans le dossier data/raw/. Le traitement Spark et la modélisation sont faits dans le second notebook, traitement_spark_ml.ipynb.

In [ ]:
%pip install python-dotenv

1. Collecte via l'API France Travail

L'API Offres d'emploi v2 de France Travail demande une authentification OAuth2 : on échange un identifiant client et un secret contre un jeton d'accès temporaire, à joindre ensuite à chaque requête. Les identifiants sont lus dans un fichier .env, exclu du dépôt Git par le .gitignore, pour ne jamais les publier.

In [ ]:
# Lecture des identifiants de l'API depuis le fichier .env (jamais écrits en clair dans le code)
from dotenv import load_dotenv
import os
load_dotenv(dotenv_path=os.path.join(os.getcwd(), ".env"))
client_id = os.getenv("CLIENT_ID")
client_secret = os.getenv("CLIENT_SECRET")

In [ ]:
import requests

In [ ]:
# Obtention du jeton d'accès OAuth2 (flux "client credentials")
URL = "https://entreprise.francetravail.fr/connexion/oauth2/access_token?realm=/partenaire"

def get_token() :
    response = requests.post(URL, data = {
        "grant_type" : "client_credentials",
        "client_id" : client_id,
        "client_secret" : client_secret,
        "scope" : "api_offresdemploiv2 o2dsoffre"
    }, headers = {"Content-Type": "application/x-www-form-urlencoded"}, verify = False)

    response.raise_for_status()
    token = response.json()["access_token"]
    print(f" Début du token: {token[:40]}")
    return token


TOKEN = get_token()


In [ ]:
# Recherche d'offres : l'API renvoie au plus 150 offres par appel, la plage est donnée par le paramètre "range"
API_URL = "https://api.francetravail.io/partenaire/offresdemploi/v2/offres/search"

def fetch_offres(token, mots_cles, start = 0, end = 4):
    headers = {
        "Authorization": f"Bearer {token}",
        "Accept" : "application/json"
    }

    params = {
        "motsCles": mots_cles,
        "range": f"{start}-{end}"
    }

    r = requests.get(API_URL, headers = headers, params = params)
    r.raise_for_status()
    return r.json()

# Test rapide sur 5 offres pour vérifier que l'authentification et la requête fonctionnent
test = fetch_offres(TOKEN, "data analyste")
offres = test.get("resultats", [])
print(f"Nombre d'offres reçues: {len(offres)}")
for o in offres:
    print(f" -{o["intitule"]} | {o.get('lieuTravail', {}).get('libelle', '?')} | {o.get('salaire', {}).get('libelle', 'salaire non renseigné')}")

In [ ]:
import json, time, os
from tqdm import tqdm
from datetime import datetime

# Mots-clés couvrant les principaux métiers de la data
mots_cles = ["data analyst", "data engineer", "data scientist", "machine learning", "analyste données", "analyste quantitatif"]

OUTPUT_DIR = "data/raw"
os.makedirs(OUTPUT_DIR, exist_ok = True)

toutes_offres = []   # toutes les offres brutes, au format JSON renvoyé par l'API

TOKEN = get_token()

for mot_cle in mots_cles :
    print(f"\nRécupération : '{mot_cle}'")
    offres_recherchees = []
    start = 0
    page = 1

    # Pagination : on avance de 150 en 150 jusqu'à une page incomplète ou vide
    while True:
        end = start + 149

        try: 
            result = fetch_offres(TOKEN, mot_cle, start = start, end = end)
            offres = result.get("resultats", [])

            if not offres:
                print(f"Page {page}: vide, arrêt.")
                break

            # on garde la trace du mot-clé et de la date de collecte
            for o in offres :
                o["_type_de_poste"] = mot_cle
                o["_heure_de_collecte"] = datetime.now().isoformat()

            offres_recherchees.extend(offres)
            print(f"Page {page} : +{len(offres)} offres (total'{mot_cle}' : {len(offres_recherchees)})")

            if len(offres) < 150:
                print(f"Dernière page atteinte")
                break

            start += 150
            page += 1
            time.sleep(1)   # pause pour respecter les quotas de l'API

        except requests.HTTPError as e:
            print(f" Erreur HTTP page {page} : {e}")
            break
        
    print(f" Total pour '{mot_cle}': {len(offres_recherchees)} offres")
    toutes_offres.extend(offres_recherchees)

print(f"\n{'='*45}")
print(f"Total Général: {len(toutes_offres)} offres collectées")
print(f"{'='*45}")

Mise en forme des offres France Travail

Chaque offre renvoyée par l'API est un objet JSON imbriqué (salaire, lieu de travail, entreprise, listes de compétences, de formations et de langues). La fonction flatten le transforme en une ligne plate de 20 colonnes, le même schéma que pour WTTJ. Les doublons (une offre trouvée par plusieurs mots-clés) sont retirés sur l'identifiant.

In [ ]:
%pip install pandas

In [ ]:
import pandas as pd
import re

# Extrait le minimum et le maximum d'un libellé du type "Annuel de 40000.0 Euros à 47000.0 Euros sur 12 mois"
def extraire_salaire(libelle):
    if not libelle:
        return None, None
    # on retire la partie "sur 12.0 mois" / "sur 13 mois" pour ne garder que les montants
    libelle = re.sub(r"sur\s+[\d.]+\s+mois", "", libelle)
    nombres = re.findall(r'\d+(?:\.\d+)?', libelle)
    nombres = [float(n) for n in nombres]
    if len(nombres) == 0:
        return None, None
    elif len(nombres) == 1:
        return nombres[0], nombres[0]
    else:
        return nombres[0], nombres[1]

# Aplatit une offre JSON en un dictionnaire d'une ligne
def flatten(o):
    #Salaire
    salaire = o.get("salaire", {})
    salaire_lib = salaire.get("libelle")
    salaire_min, salaire_max = extraire_salaire(salaire_lib)

    #Lieu de travail
    lieu = o.get("lieuTravail", {})
    ville = lieu.get("libelle")
    code_postal = lieu.get("codePostal")
    commune = lieu.get("commune")

    #Entreprise
    entreprise = o.get("entreprise", {})
    nom_entreprise = entreprise.get("nom")
    secteur_entreprise = o.get("secteurActiviteLibelle")

    #Competence
    competences = o.get("competences", [])
    skills = ", ".join([c.get("libelle", "") for c in competences])

    #Formation requise
    formations = o.get("formations", [])
    niveaux_formation = ", ".join([f.get("niveauLibelle", "") for f in formations])

    #Langues requise
    langues = o.get("langues", [])
    langues_str = ", ".join([l.get("libelle", "") for l in langues])


    return {
        "id": o.get("id"),
        "intitule": o.get("intitule"),
        "description": o.get("description"),
        "type_contrat": o.get("typeContratLibelle"),
        "nb_heure_travail": o.get("dureeTravailLibelle"),
        "experience": o.get("experienceLibelle"),
        "experience_exige": o.get("experienceExige"),
        "salaire_libelle": salaire_lib,
        "salaire_min" : salaire_min,
        "salaire_max": salaire_max,
        "ville": ville,
        "code_postal": code_postal,
        "commune": commune,
        "entreprise": nom_entreprise,
        "secteur_activite": secteur_entreprise,
        "competences": skills,
        "formation_requise": niveaux_formation,
        "langues": langues_str,
        'type_de_poste': o.get("_type_de_poste"),
        "source": "France travail" 

    }

#Application de la fonction sur toute les offres
offres_extraites = [flatten(o) for o in toutes_offres]
df = pd.DataFrame(offres_extraites)
df = df.drop_duplicates(subset = "id").reset_index(drop = True)
print(f"Shape : {df.shape[0]} offres et {df.shape[1]} colonnes")
df.head()

Sauvegarde des données brutes

On enregistre deux fichiers dans data/raw/ : la réponse brute de l'API en JSON, pour pouvoir refaire tout le traitement sans rappeler l'API, et la version aplatie en CSV, qui sera lue par Spark.

In [ ]:
# Sauvegarde des données brutes (zone "raw" du data lake)
#  - JSON : réponse brute de l'API, sans aucune transformation (rejouable)
#  - CSV  : version aplatie, même schéma que les offres WTTJ
date_collecte = datetime.now().strftime("%Y%m%d")
with open(os.path.join(OUTPUT_DIR, f"france_travail_{date_collecte}.json"), "w", encoding="utf-8") as f:
    json.dump(toutes_offres, f, ensure_ascii=False)

df.to_csv(os.path.join(OUTPUT_DIR, "offres_france_travail.csv"), index=False, encoding="utf-8-sig")
print(f"{len(df)} offres France Travail sauvegardées dans {OUTPUT_DIR}/")

2. Web scraping de Welcome to the Jungle

WTTJ n'a pas d'API publique et ses pages sont construites en JavaScript : un simple requests ne renvoie pas les offres, et les pages de détail sont protégées contre les robots. On pilote donc un navigateur Chrome sans interface avec Selenium, puis on analyse le HTML obtenu avec BeautifulSoup.

Depuis notre première collecte d'avril 2026, le site a changé : la page de résultats de recherche n'est plus accessible sans inscription, et les anciens sélecteurs HTML ne trouvaient plus aucune offre. La liste des offres est en revanche servie au navigateur par le moteur de recherche Algolia, avec une clé publique de recherche que le site publie lui-même. Le scraper fonctionne donc en deux temps :
1. pour chaque mot-clé, on interroge cette recherche Algolia (offres situées en France, les plus pertinentes d'abord) : on obtient l'intitulé complet, l'entreprise, le contrat, la ville, le salaire, l'expérience et le niveau d'études ;
2. pour chaque offre, on ouvre sa page de détail avec Selenium pour lire la description complète et le code postal dans les données structurées JSON-LD, un format standard que les sites publient pour les moteurs de recherche.

Pour rester raisonnable vis-à-vis du site, on garde au plus 250 offres par mot-clé et 1500 au total, avec une pause de 1 à 2 secondes entre deux requêtes. Les libellés (salaire, expérience, formation) sont mis au format de ceux de France Travail pour faciliter le traitement commun dans le second notebook.

In [ ]:
pip install beautifulsoup4

In [ ]:
%pip install selenium webdriver_manager

In [ ]:
%pip install chromedriver-binary-auto

In [ ]:
import json, os, re, time, random
import requests
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

mots_cles = ["data analyst", "data engineer", "data scientist", "machine learning", "analyste données", "analyste quantitatif"]
OUTPUT_DIR = "data/raw"
os.makedirs(OUTPUT_DIR, exist_ok = True)

URL_WTTJ = "https://www.welcometothejungle.com"
INDEX_OFFRES = "wttj_jobs_production_fr"   # index Algolia des offres en français
MAX_PAR_MOT_CLE = 250                       # offres retenues par mot-clé (les plus pertinentes)
MAX_OFFRES = 1500                           # plafond global après dédoublonnage
CHEMIN_CHROMEDRIVER = r"C:\chromedriver\chromedriver-win64\chromedriver.exe"

# Lance un Chrome sans interface (headless) piloté par Selenium
def init_driver():
    options = webdriver.ChromeOptions()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/155.0 Safari/537.36")
    options.page_load_strategy = "eager"   # on n'attend pas les images et les publicités
    try:
        service = Service(executable_path = CHEMIN_CHROMEDRIVER)
        driver = webdriver.Chrome(service = service, options = options)
    except Exception:
        # chromedriver local incompatible avec la version de Chrome installée :
        # Selenium Manager (inclus dans Selenium) récupère alors automatiquement le bon pilote
        driver = webdriver.Chrome(options = options)
    return driver


# Le site ne propose plus de page de résultats publique (la recherche redirige vers une inscription).
# Les offres sont servies par le moteur de recherche Algolia, dont l'identifiant et la clé publique de recherche
# sont publiés par le site lui-même dans /api/env : on les lit depuis le navigateur plutôt que de les écrire en dur.
def get_config_algolia(driver):
    driver.get(f"{URL_WTTJ}/fr/jobs")
    time.sleep(3)
    env = driver.execute_script("return fetch('/api/env').then(r => r.text())")
    env = json.loads(env[env.index("{"): env.rindex("}") + 1])
    return env["PUBLIC_ALGOLIA_APPLICATION_ID"], env["PUBLIC_ALGOLIA_API_KEY_CLIENT"]


# Interroge l'index des offres pour un mot-clé (offres situées en France, triées par pertinence)
def fetch_page_wttj(app_id, api_key, mot_cle, page = 0, nb_par_page = 50):
    url = f"https://{app_id.lower()}-dsn.algolia.net/1/indexes/{INDEX_OFFRES}/query"
    headers = {
        "X-Algolia-Application-Id": app_id,
        "X-Algolia-API-Key": api_key,
        "Referer": f"{URL_WTTJ}/",
        "Origin": URL_WTTJ
    }
    params = {"query": mot_cle, "hitsPerPage": nb_par_page, "page": page, "filters": "offices.country_code:FR"}
    r = requests.post(url, headers = headers, json = params)
    r.raise_for_status()
    return r.json()


# Correspondances entre les codes WTTJ et des libellés proches de ceux de France Travail
CONTRATS = {"full_time": "CDI", "temporary": "CDD / Temporaire", "internship": "Stage", "apprenticeship": "Alternance",
            "freelance": "Freelance", "vie": "VIE", "part_time": "Temps partiel", "other": "Autres"}
FORMATIONS = {"no_diploma": "Sans diplôme", "bac": "Bac", "bac_2": "Bac+2", "bac_3": "Bac+3", "bac_4": "Bac+4",
              "bac_5": "Bac+5", "phd": "Doctorat"}
PERIODES = {"yearly": "Annuel", "monthly": "Mensuel", "daily": "Journalier", "hourly": "Horaire"}

def texte_html(html):
    return BeautifulSoup(html, "html.parser").get_text(separator = " ", strip = True) if html else None


# Libellé du salaire au même format que France Travail : "Annuel de 45000 Euros à 50000 Euros"
def libelle_salaire(o):
    s_min, s_max = o.get("salary_minimum"), o.get("salary_maximum")
    if not s_min and not s_max:
        return None
    periode = PERIODES.get(o.get("salary_period"), "Annuel")
    devise = o.get("salary_currency") or "EUR"
    devise = "Euros" if devise == "EUR" else devise
    if s_min and s_max and s_min != s_max:
        return f"{periode} de {s_min:.0f} {devise} à {s_max:.0f} {devise}"
    return f"{periode} de {(s_min or s_max):.0f} {devise}"


# Expérience minimale au format France Travail ("3 An(s)", "6 Mois", "Débutant accepté")
def libelle_experience(annees):
    if annees is None:
        return None, None
    if annees == 0:
        return "Débutant accepté", "D"
    if annees < 1:
        return f"{round(annees * 12)} Mois", "E"
    return f"{annees:g} An(s)", "E"


# Transforme un résultat Algolia en ligne de 20 colonnes (la description et le code postal sont complétés ensuite)
def parser_offre(o, mot_cle):
    bureaux = o.get("offices") or [{}]
    bureau = next((b for b in bureaux if b.get("country_code") == "FR"), bureaux[0])
    experience, experience_exige = libelle_experience(o.get("experience_level_minimum"))
    entreprise = o.get("organization") or {}
    return {
        "id": o.get("slug"),
        "intitule": (o.get("name") or "").strip() or None,
        "description": None,
        "type_contrat": CONTRATS.get(o.get("contract_type"), o.get("contract_type")),
        "nb_heure_travail": None,
        "experience": experience,
        "experience_exige": experience_exige,
        "salaire_libelle": libelle_salaire(o),
        "salaire_min": o.get("salary_minimum"),
        "salaire_max": o.get("salary_maximum"),
        "ville": bureau.get("city"),
        "code_postal": None,
        "commune": bureau.get("district"),
        "entreprise": entreprise.get("name"),
        "secteur_activite": ", ".join(s["name"] for s in o.get("sectors") or [] if s.get("name")) or None,
        "competences": None,
        "formation_requise": FORMATIONS.get(o.get("education_level"), o.get("education_level")),
        "langues": None,
        "type_de_poste": mot_cle,
        "source": "welcometothejungle",
        # champs techniques, retirés avant la sauvegarde
        "_url": f"{URL_WTTJ}/fr/companies/{entreprise.get('slug')}/jobs/{o.get('slug')}",
        "_resume": " ".join(filter(None, [o.get("summary"), " ".join(o.get("key_missions") or []), texte_html(o.get("profile"))]))
    }


# Ouvre la page de détail d'une offre et lit ses données structurées JSON-LD (description complète, code postal)
def completer_offre(driver, offre):
    try:
        driver.get(offre["_url"])
        WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.CSS_SELECTOR, 'script[type="application/ld+json"]')))
        soup_detail = BeautifulSoup(driver.page_source, "html.parser")

        for tag in soup_detail.find_all("script", type = "application/ld+json"):
            data = json.loads(tag.string or "{}")
            if not isinstance(data, dict) or data.get("@type") != "JobPosting":
                continue
            offre["description"] = texte_html(data.get("description"))
            if data.get("title"):
                offre["intitule"] = data["title"].strip()
            if data.get("hiringOrganization", {}).get("name"):
                offre["entreprise"] = data["hiringOrganization"]["name"]

            locations = data.get("jobLocation", [])
            if isinstance(locations, dict):
                locations = [locations]
            adresses = [l.get("address", {}) for l in locations]
            adresse = next((a for a in adresses if a.get("addressCountry") == "FR"), adresses[0] if adresses else {})
            offre["code_postal"] = adresse.get("postalCode")
            offre["commune"] = offre["commune"] or adresse.get("addressRegion")
            offre["ville"] = offre["ville"] or adresse.get("addressLocality")
            break

        # le "Profil recherché" n'est pas toujours repris dans le JSON-LD : on l'ajoute à la description
        profil_tag = soup_detail.find(attrs = {"data-testid": "job-section-experience"})
        profil = profil_tag.get_text(separator = " ", strip = True) if profil_tag else None
        if profil and offre["description"] and profil[:80] not in offre["description"]:
            offre["description"] = f"{offre['description']} {profil}"
    except Exception as e:
        print(f" Erreur détail '{offre['id']}' : {type(e).__name__}")

    if offre["description"]:
        offre["description"] = re.sub(r"\s*Voir plus$", "", offre["description"])

    # à défaut de page de détail, on garde le résumé fourni par Algolia (résumé, missions, profil)
    if not offre["description"]:
        offre["description"] = offre["_resume"] or None
    return offre


toutes_offres_wttj = []
driver = init_driver()
try:
    app_id, api_key = get_config_algolia(driver)

    # 1. Liste des offres par mot-clé, via la recherche Algolia du site
    for mot_cle in mots_cles :
        print(f"\nRécupération : '{mot_cle}'")
        offres_mot_cle = []
        page = 0
        while len(offres_mot_cle) < MAX_PAR_MOT_CLE:
            try:
                result = fetch_page_wttj(app_id, api_key, mot_cle, page = page)
            except requests.HTTPError as e:
                print(f" Erreur HTTP page {page + 1} : {e}")
                break
            hits = result.get("hits", [])
            if not hits:
                break
            offres_mot_cle.extend(parser_offre(o, mot_cle) for o in hits)
            if page == 0:
                print(f" {result.get('nbHits')} offres en France sur WTTJ")
            page += 1
            if page >= result.get("nbPages", 0):
                break
            time.sleep(random.uniform(1, 2))
        offres_mot_cle = offres_mot_cle[:MAX_PAR_MOT_CLE]
        print(f" Total pour '{mot_cle}': {len(offres_mot_cle)} offres retenues")
        toutes_offres_wttj.extend(offres_mot_cle)

    # une même offre peut ressortir pour plusieurs mots-clés : on garde la première occurrence
    vus = set()
    toutes_offres_wttj = [o for o in toutes_offres_wttj if o["id"] and not (o["id"] in vus or vus.add(o["id"]))][:MAX_OFFRES]
    print(f"\n{len(toutes_offres_wttj)} offres uniques à compléter depuis leur page de détail")

    # 2. Page de détail de chaque offre, avec une pause de 1 à 2 s entre deux pages
    for i, offre in enumerate(toutes_offres_wttj, start = 1):
        # Chrome finit par saturer sur une longue session : on le relance toutes les 200 pages,
        # ou immédiatement si la session a été perdue
        try:
            session_ok = i % 200 != 0 and bool(driver.window_handles)
        except Exception:
            session_ok = False
        if not session_ok:
            try:
                driver.quit()
            except Exception:
                pass
            driver = init_driver()
        completer_offre(driver, offre)
        if i % 100 == 0 or i == len(toutes_offres_wttj):
            print(f" {i}/{len(toutes_offres_wttj)} pages de détail visitées")
        time.sleep(random.uniform(1, 2))
except Exception as e:
    print(f"Erreur générale : {e}")
finally:
    driver.quit()

df_wttj = pd.DataFrame(toutes_offres_wttj).drop(columns = ["_url", "_resume"], errors = "ignore")
df_wttj = df_wttj.drop_duplicates(subset = "id").reset_index(drop = True)
print(f"\n{'='*45}")
print(f"Shape : {df_wttj.shape[0]} offres et {df_wttj.shape[1]} colonnes")
print(f"{'='*45}")
print(df_wttj["type_de_poste"].value_counts().to_string())
df_wttj[["intitule", "entreprise", "ville", "type_contrat", "salaire_libelle"]].head()

Sauvegarde

Les offres WTTJ sont enregistrées dans data/raw/, à côté de celles de France Travail, avec le même schéma de colonnes. Une copie est aussi écrite à la racine du projet, à la place du fichier de la collecte d'avril.

In [ ]:
df_wttj.to_csv(os.path.join(OUTPUT_DIR, "offres_wttj.csv"), index = False, encoding = "utf-8-sig")
df_wttj.to_csv("offres_wttj.csv", index = False, encoding = "utf-8-sig")
print(f"{len(df_wttj)} offres WTTJ sauvegardées dans {OUTPUT_DIR}/ et à la racine du projet")